In [0]:
# Create (first time) or update (afterwards) the model serving endpoint so it serves the registered champion.
# Runs in every environment (dev / test / prod). The endpoint name is the configured base name in prod and
# '<base>-<target>' elsewhere, so environments sharing a workspace never collide.
import sys
import os
sys.path.insert(0, os.getcwd())

import mlflow
from mlflow import MlflowClient
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput
from utils import load_project, model_name as qualified_model_name, widget_or

mlflow.set_registry_uri("databricks-uc")
cfg = load_project()
sv = cfg["serving"]

dbutils.widgets.text("catalog", "stock_demo")
dbutils.widgets.text("target", "dev")
dbutils.widgets.text("endpoint_name", "")
catalog = dbutils.widgets.get("catalog")
target = dbutils.widgets.get("target")
custom_name = str(widget_or(dbutils.widgets.get("endpoint_name"), "") or "").strip()
base_name = sv["endpoint_name"]
# explicit widget wins; otherwise prod -> stock-price-regressor, dev -> stock-price-regressor-dev, test -> ...-test
endpoint_name = custom_name or (base_name if target == "prod" else f"{base_name}-{target}")
print(f"Target: {target} | endpoint: {endpoint_name}")

registered_model_name = qualified_model_name(cfg, catalog)
client = MlflowClient()
champion_mv = client.get_model_version_by_alias(registered_model_name, "champion")
champion_version = str(champion_mv.version)
print(f"Champion model version: {champion_version}")

w = WorkspaceClient()
served = [ServedEntityInput(
    entity_name=registered_model_name,
    entity_version=champion_version,
    scale_to_zero_enabled=bool(sv["scale_to_zero"]),
    workload_size=sv["workload_size"],
)]

try:
    endpoint = w.serving_endpoints.get(name=endpoint_name)
    print(f"Endpoint exists: {endpoint.name} | ready state: {endpoint.state.ready}")
    w.serving_endpoints.update_config(name=endpoint_name, served_entities=served)
    print(f"Endpoint '{endpoint_name}' update submitted.")
except NotFound:
    print(f"Endpoint '{endpoint_name}' not found - creating it (Model Serving must be enabled in this workspace).")
    w.serving_endpoints.create(name=endpoint_name, config=EndpointCoreConfigInput(name=endpoint_name, served_entities=served))
    print(f"Endpoint '{endpoint_name}' creation submitted. It becomes READY in a few minutes (check the Serving page).")

print(f"Now serving champion version {champion_version} (100%).")